# Genotype–Phenotype Heterogeneity Exploration with `mlcroissant`
This notebook demonstrates loading and processing the FAIR² dataset using the `mlcroissant` library, focusing on clinical, hormonal, imaging, and genetic data from three female patients with non-classical 21-Hydroxylase Deficiency-associated infertility.

### Dataset Source
The dataset is structured via a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load dataset metadata and initialize objects for exploration.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.cbsv-djdb/fair2.json'

# Load dataset
dataset = mlc.Dataset(croissant_url)

# Access dataset metadata attributes
print(f"Name: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}")
print(f"Authors: {dataset.metadata.author}")
print(f"Data Biases: {dataset.metadata.dataBiases}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

- The dataset contains multiple record sets (tables), each with uniquely defined fields/columns.
- Record sets are referenced via their `@id` (e.g. `cr:recordSet`).
- Fields and columns are also referenced via their `@id`.

Let's enumerate record sets and their contents.

In [ ]:
# List available record sets and their fields
record_sets = dataset.record_sets
print("Record sets found:")
for rs in record_sets:
    print(f"- @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    print("  Fields:")
    for f in fields:
        print(f"    - @id: {f['@id']} | Name: {f.get('name', 'N/A')}")
print("\nPreviewing sample records from each set:")

# Preview records using @id reference
for rs in record_sets:
    rs_id = rs['@id']
    print(f"\nRecords for RecordSet {rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            print(rec)
            if i >= 1:
                break
    except Exception as e:
        print(f"Could not load preview for {rs_id}: {e}")

## 3. Data Extraction
Load each record set
- Use their `@id` values from the previous overview.
- Place the extracted records for each set in a pandas DataFrame.

In [ ]:
dataframes = {}
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"RecordSet @id list: {record_sets_ids}")

for rs_id in record_sets_ids:
    # Load all records for this record set
    try:
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\n--- DataFrame columns for {rs_id} ---")
        print(df.columns.tolist())
        print(df.head())
    except Exception as e:
        print(f"Could not load DataFrame for {rs_id}: {e}")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

For demonstration, let's:
- Filter for age at diagnosis greater than a threshold.
- Normalize height.
- Group by presence of infertility.

We will use the `@id` for relevant fields.

In [ ]:
# Example with first record set (clinical features)
# Replace the following values with real @id's based on the actual schema inspection
clinical_rs_id = record_sets_ids[0]
df = dataframes[clinical_rs_id]

# Example field @id's (replace as appropriate)
age_id = df.columns[df.columns.str.contains("age")].tolist()[0] if df.columns.str.contains("age").any() else df.columns[0]
height_id = df.columns[df.columns.str.contains("height")].tolist()[0] if df.columns.str.contains("height").any() else df.columns[1]
infertility_id = df.columns[df.columns.str.contains("infertility")].tolist()[0] if df.columns.str.contains("infertility").any() else df.columns[-1]

# Filtering: age at diagnosis > 25
threshold = 25
filtered_df = df[df[age_id] > threshold]
print(f"Filtered records with {age_id} > {threshold}:")
print(filtered_df.head())

# Normalize height
filtered_df[f"{height_id}_normalized"] = (filtered_df[height_id] - filtered_df[height_id].mean()) / filtered_df[height_id].std()
print(f"Normalized {height_id} for filtered records:")
print(filtered_df[[height_id, f"{height_id}_normalized"]].head())

# Group by infertility status
if infertility_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(infertility_id).mean(numeric_only=True)
    print(f"Grouped data by {infertility_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize distributions for numeric fields, such as height or hormone levels.

Let's plot normalized height for filtered patients.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 4))
plt.hist(filtered_df[f"{height_id}_normalized"], bins=6, color='skyblue', edgecolor='black')
plt.xlabel('Normalized Height')
plt.ylabel('Count')
plt.title('Distribution of Normalized Height (Age > 25)')
plt.grid(axis='y', alpha=0.7)
plt.tight_layout()
plt.show()

## 6. Conclusion
- The dataset structure and field IDs allow precise exploration across clinical, hormonal, imaging, and genetic factors.
- Filtering, normalization, and grouping can be performed efficiently referencing entities by their `@id`.
- The small cohort illustrates genotype–phenotype relationships, but limitations include N=3 and single-center data.

_For further analysis, consult the Croissant schema field `@id`s and extend EDA or modeling as appropriate._